In [ ]:
import os
import time
import math
import re
from datetime import datetime
import requests
from dotenv import load_dotenv
import pandas as pd

#API 호출 및 선택 컬럼 수집
load_dotenv()
DATA_EX_KEY = os.getenv("DATA_EX_KEY")
if not DATA_EX_KEY:
    raise RuntimeError("환경변수 DATA_EX_KEY가 설정되어 있지 않습니다.")

base_url = "https://data.ex.co.kr/openapi/restinfo/restEventList"
per_page = 100
params = {"key": DATA_EX_KEY, "type": "json", "numOfRows": per_page, "pageNo": 1}

# 호출로 총 개수 확인
r0 = requests.get(base_url, params=params, timeout=10)
r0.raise_for_status()
j0 = r0.json()
if j0.get("code") == "ERROR":
    raise RuntimeError(f"API error: {j0.get('message')}")
total_count = j0.get("count", 0)
if not total_count:
    df_all = pd.DataFrame()
else:
    total_pages = math.ceil(total_count / per_page)
    want_cols = ["stdRestCd", "stime", "routeNm", "stdRestNm", "etime", "eventDetail", "eventNm"]
    frames = []
    for page in range(1, total_pages + 1):
        params["pageNo"] = page
        r = requests.get(base_url, params=params, timeout=10)
        r.raise_for_status()
        j = r.json()
        if j.get("code") == "ERROR":
            raise RuntimeError(f"API error on page {page}: {j.get('message')}")
        page_list = j.get("list", [])
        if page_list:
            df_page = pd.json_normalize(page_list)
            df_page.columns = df_page.columns.str.strip()
            selected = [c for c in want_cols if c in df_page.columns]
            if selected:
                df_sel = df_page[selected].copy()
                df_sel["_page"] = page
                frames.append(df_sel)
        time.sleep(0.15)
    df_all = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=want_cols + ["_page"])

#컬럼명 정리 및 전처리
rename_map = {
    "stdRestCd": "std_rest_cd",
    "stime": "start_time",
    "routeNm": "route_name",
    "stdRestNm": "rest_name",
    "etime": "end_time",
    "eventDetail": "event_detail",
    "eventNm": "event_name"
}
existing_rename = {k: v for k, v in rename_map.items() if k in df_all.columns}
df = df_all.rename(columns=existing_rename)

for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].astype(str).str.strip().replace({"nan": pd.NA, "None": pd.NA})

# 안전하게 end_time 컬럼 확보
if "end_time" not in df.columns:
    raise KeyError("end_time(etime) 컬럼이 존재하지 않습니다. API 응답 컬럼명을 확인하세요.")
df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")


# end_time 기준 기본 필터 (>= 2026-03-29)
cutoff = pd.Timestamp("2026-03-29")
df_after_cutoff = df[df["end_time"].notna() & (df["end_time"] >= cutoff)].reset_index(drop=True)

-
# event_detail에서 날짜 추출 함수
date_patterns = [
    # full year-month-day with separators
    r'(?P<y>(19|20)\d{2})[.\-/](?P<m>0?[1-9]|1[0-2])[.\-/](?P<d>0?[1-9]|[12][0-9]|3[01])',
    # Korean style: 2026년 3월 29일 or 2026년03월29일
    r'(?P<y2>(19|20)\d{2})\s*년\s*(?P<m2>0?[1-9]|1[0-2])\s*월\s*(?P<d2>0?[1-9]|[12][0-9]|3[01])\s*일',
    # month/day without year (MM/DD or MM.DD or MM-DD)
    r'(?P<m3>0?[1-9]|1[0-2])[./-](?P<d3>0?[1-9]|[12][0-9]|3[01])',
    # Korean month/day without year: 3월 29일
    r'(?P<m4>0?[1-9]|1[0-2])\s*월\s*(?P<d4>0?[1-9]|[12][0-9]|3[01])\s*일'
]

def extract_dates_from_text(text):
    """
    텍스트에서 날짜들을 datetime 리스트로 반환.
    연도가 명시되지 않은 경우 2026년으로 간주(사용자 요구에 따라).
    """
    if not isinstance(text, str) or pd.isna(text):
        return []
    found = []
    s = text
    # 1) YYYY-MM-DD 등
    for pat in date_patterns:
        for m in re.finditer(pat, s):
            gd = m.groupdict()
            try:
                if gd.get("y"):
                    y = int(gd["y"])
                    mth = int(gd["m"])
                    day = int(gd["d"])
                    found.append(datetime(y, mth, day))
                elif gd.get("y2"):
                    y = int(gd["y2"])
                    mth = int(gd["m2"])
                    day = int(gd["d2"])
                    found.append(datetime(y, mth, day))
                elif gd.get("m3"):
                    # no year -> assume 2026
                    y = 2026
                    mth = int(gd["m3"])
                    day = int(gd["d3"])
                    found.append(datetime(y, mth, day))
                elif gd.get("m4"):
                    y = 2026
                    mth = int(gd["m4"])
                    day = int(gd["d4"])
                    found.append(datetime(y, mth, day))
            except ValueError:
                # invalid date like 2026-02-30 등은 무시
                continue
    # 중복 제거 및 정렬
    found = sorted(set(found))
    return found

# event_detail 기반 추가 필터링
def should_exclude_by_event_detail(event_detail_text, cutoff_dt=cutoff.to_pydatetime()):
    """
    event_detail 텍스트에 날짜가 명시되어 있으면,
    - 연도가 2026보다 작으면 True (제외)
    - 연도가 2026이고 날짜가 cutoff 이전이면 True (제외)
    - 연도 미포함으로 월/일만 있으면 2026년으로 간주해 cutoff 이전이면 True
    그 외(날짜 없음 또는 모두 cutoff 이후)는 False (제외하지 않음)
    """
    dates = extract_dates_from_text(event_detail_text)
    if not dates:
        return False  # 날짜 정보가 없으면 제외하지 않음(원칙: 텍스트로 과거 행사 명시가 없으면 유지)
    for d in dates:
        if d.year < 2026:
            return True
        if d.year == 2026 and d < cutoff_dt:
            return True
    return False

# 적용: df_after_cutoff에서 event_detail 검사하여 제외
mask_exclude = df_after_cutoff["event_detail"].apply(lambda t: should_exclude_by_event_detail(t))
df_final = df_after_cutoff[~mask_exclude].reset_index(drop=True)


# 결과 출력
print("원본(전체) 행수:", len(df))
print("end_time 기준 필터(>= 2026-03-29) 후 행수:", len(df_after_cutoff))
print("event_detail 텍스트로 추가 제외된 행수:", mask_exclude.sum())
print("최종 남은 행수:", len(df_final))
print("\n최종 샘플 (상위 50개):")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(df_final.head(50).to_string(index=False))

In [ ]:
import os
from urllib.parse import quote_plus
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd
import unicodedata

# 설정 및 환경변수 로드
load_dotenv()
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")
DB_PORT = os.getenv("DB_PORT", "3306")

if not (DB_HOST and DB_USER and DB_PASSWORD and DB_NAME):
    raise RuntimeError("DB_HOST, DB_USER, DB_PASSWORD, DB_NAME 환경변수를 모두 설정하세요.")

# df_final 존재 확인
try:
    df_final
except NameError:
    raise RuntimeError("df_final 변수가 존재하지 않습니다. 먼저 API 추출 및 전처리 코드를 실행해 주세요.")

# 정규화 함수
def normalize_name(s):
    if s is None:
        return None
    if not isinstance(s, str):
        s = str(s)
    s = unicodedata.normalize("NFKC", s).strip()
    s = " ".join(s.split())
    return s if s != "" else None

# DataFrame 준비 및 전처리
df = df_final.copy()

# rest_name -> restarea_name 매핑
if "rest_name" in df.columns and "restarea_name" not in df.columns:
    df = df.rename(columns={"rest_name": "restarea_name"})

# 목표 컬럼
target_cols = ["std_rest_cd", "start_time", "route_name", "restarea_name", "end_time", "event_detail", "event_name"]
present_cols = [c for c in target_cols if c in df.columns]
df = df[present_cols].copy()

# 날짜 타입 정리
for dt_col in ["start_time", "end_time"]:
    if dt_col in df.columns:
        df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")

# 문자열 정리 및 정규화
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).replace({"nan": None, "None": None})
    df[c] = df[c].apply(lambda x: normalize_name(x) if x is not None else None)

# NaN -> None
df = df.where(pd.notnull(df), None)

# DB 연결 준비
pwd_quoted = quote_plus(DB_PASSWORD)
mysql_url = f"mysql+pymysql://{DB_USER}:{pwd_quoted}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
engine = create_engine(mysql_url, echo=False, pool_pre_ping=True)

# 연결 테스트
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))


# rest_area_events 테이블 생성(없으면) — 외래키 포함 가능
create_table_sql = """
CREATE TABLE IF NOT EXISTS rest_area_events (
  id BIGINT AUTO_INCREMENT PRIMARY KEY,
  std_rest_cd VARCHAR(100),
  start_time DATETIME,
  route_name VARCHAR(255),
  restarea_name VARCHAR(255),
  end_time DATETIME,
  event_detail TEXT,
  event_name VARCHAR(255),
  UNIQUE KEY uq_std_rest_cd (std_rest_cd)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci;
"""
with engine.begin() as conn:
    conn.execute(text(create_table_sql))

# 삽입/업서트 준비
if "std_rest_cd" not in df.columns:
    raise RuntimeError("std_rest_cd 컬럼이 필요합니다. 업서트 키로 사용됩니다.")

df_insert = df[df["std_rest_cd"].notna()].copy()
if df_insert.empty:
    print("삽입할 유효한 행(std_rest_cd 존재)이 없습니다. 종료합니다.")
else:
    insert_cols = [c for c in ["std_rest_cd", "start_time", "route_name", "restarea_name", "end_time", "event_detail", "event_name"] if c in df_insert.columns]
    col_list_sql = ", ".join([f"`{c}`" for c in insert_cols])
    placeholders = ", ".join(["%s"] * len(insert_cols))
    update_assignments = ", ".join([f"`{c}`=VALUES(`{c}`)" for c in insert_cols if c != "std_rest_cd"])
    insert_sql = f"INSERT INTO rest_area_events ({col_list_sql}) VALUES ({placeholders})"
    if update_assignments:
        insert_sql += f" ON DUPLICATE KEY UPDATE {update_assignments}"

    chunk_size = 1000
    total = len(df_insert)
    inserted = 0

    # 외래키 검사 비활성화(세션 단위), 삽입 수행, 복구
    # 예외가 발생하더라도 finally에서 FK 검사를 복구하도록 함
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur:
            # 세션에서 외래키 검사 비활성화
            cur.execute("SET SESSION FOREIGN_KEY_CHECKS=0;")
            raw_conn.commit()
            print("세션 FOREIGN_KEY_CHECKS=0 설정: 외래키 검사 비활성화")

            # 청크 단위 bulk insert
            for start in range(0, total, chunk_size):
                end = min(start + chunk_size, total)
                chunk = df_insert.iloc[start:end]
                values = []
                for _, row in chunk.iterrows():
                    tup = []
                    for c in insert_cols:
                        val = row[c]
                        if hasattr(val, "to_pydatetime"):
                            val = val.to_pydatetime()
                        tup.append(val)
                    values.append(tuple(tup))
                # bulk insert
                cur.executemany(insert_sql, values)
                raw_conn.commit()
                inserted += len(values)
                print(f"Inserted/Updated rows: {inserted}/{total}")

            print("모든 데이터 삽입/업데이트 완료 (외래키 검사 비활성화 상태에서 수행됨)")

    except Exception as e:
        raw_conn.rollback()
        # 문제 행을 찾아 출력
        print("Bulk insert 실패, 문제 행 진단을 시도합니다...")
        try:
            with raw_conn.cursor() as cur2:
                for i, row in enumerate(values):
                    try:
                        cur2.execute(insert_sql, row)
                        raw_conn.commit()
                    except Exception as ex_row:
                        raw_conn.rollback()
                        print("문제 발생한 행 인덱스(청크 내):", i)
                        print("문제 행 값:", dict(zip(insert_cols, row)))
                        print("예외:", repr(ex_row))
                        raise RuntimeError(f"삽입 중 오류 발생(문제 행 출력됨): {ex_row}")
        finally:
            # FK 검사 복구 시도
            try:
                with raw_conn.cursor() as cur3:
                    cur3.execute("SET SESSION FOREIGN_KEY_CHECKS=1;")
                    raw_conn.commit()
                    print("세션 FOREIGN_KEY_CHECKS=1 설정: 외래키 검사 복구 완료")
            except Exception as e2:
                print("외래키 검사 복구 중 오류:", e2)
        # 재발생
        raise RuntimeError(f"삽입 중 오류 발생: {e}")

    finally:
        # FK 검사 복구
        try:
            with raw_conn.cursor() as cur_final:
                cur_final.execute("SET SESSION FOREIGN_KEY_CHECKS=1;")
                raw_conn.commit()
                print("세션 FOREIGN_KEY_CHECKS=1 설정: 외래키 검사 복구 완료 (finally)")
        except Exception as e_final:
            print("외래키 검사 복구 중 오류(최종):", e_final)
        raw_conn.close()

# 잘들어갔는지 확인
with engine.connect() as conn:
    cnt = conn.execute(text("SELECT COUNT(*) FROM rest_area_events")).scalar()
    print(f"rest_area_events 테이블 총 레코드 수: {cnt}")